# Disease Sample Scoring Pipeline

| Branch | 모델 | Score | 해석 |
|--------|------|-------|------|
| NBI (det≥10%) | GAMLSS NBI | z_nbi | 발현량 이상도 (calibrated) |
| Logistic (1-10%) | Logistic Regression | z_logistic | 검출 확률 이상도 |
| Rare (det<1%) | Poisson / Fixed | rare_score | 희소 발현 이상도 (uncalibrated) |

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')   # Modeling/ → pipeline 패키지 import
sys.path.insert(0, '..')  # ../config.py 참조
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from pipeline import data_prep, scoring, plots

In [ ]:
# engine(NBI/Logistic) + rare_event_scorer 로드 (없으면 학습/구축, config 경로/파라미터)
engine, rare_scorer = scoring.load_engine()
print(f'  NBI genes : {len(engine.count_genes):,}   Logistic : {len(engine.logistic_genes):,}')

In [ ]:
# disease/HC 분리 (raw Phenotype_Processed — study-split 미적용, scoring 원본 동일)
adata      = data_prep.load_adata()
is_hc      = (adata.obs['Phenotype_Processed'].astype(str) == 'Healthy Control').values
is_dis     = ~is_hc
Y          = data_prep.count_matrix(adata)
X_raw      = data_prep.bias_matrix(adata)
sample_ids = adata.obs_names.tolist()
gene_names = adata.var_names.tolist()
phenotypes = adata.obs['Phenotype_Processed'].astype(str).values
dis_idx    = np.where(is_dis)[0]
dis_names  = [sample_ids[i] for i in dis_idx]
dis_pheno  = phenotypes[dis_idx]
print(f'HC: {is_hc.sum()}  Disease: {is_dis.sum()}')
print(pd.Series(dis_pheno).value_counts().to_string())

In [ ]:
# 단일 샘플 스코어링 래퍼 (scoring.score_all 사용, 시각화용)
def score_sample(sample_idx):
    df = scoring.score_all(engine, rare_scorer, gene_names,
                           X_raw[[sample_idx]], Y[[sample_idx]],
                           [sample_ids[sample_idx]], [phenotypes[sample_idx]], min_abs_score=0.0)
    return df.drop(columns=['sample', 'phenotype'])

print('Scorer ready.')

In [ ]:
# per-sample Manhattan 시각화 → pipeline.plots.plot_sample (z_flag = config.MODELING_PARAMS)
print('plots.plot_sample(df, sample_id, phenotype) 사용')

In [ ]:
Z_FLAG = config.MODELING_PARAMS['z_flag']
for pheno in ['Liver Cancer', 'Pancreatic Cancer', 'Tuberculosis']:
    mask = (phenotypes == pheno) & is_dis
    if not mask.any():
        print(f'{pheno}: 샘플 없음'); continue
    idx = np.where(mask)[0][1]
    sid = sample_ids[idx]
    print(f'\nScoring {pheno}: {sid}')
    df = score_sample(idx)
    flagged = df[df['score'].abs() >= Z_FLAG]
    print(f'  Flagged genes: {len(flagged)}')
    print(flagged.nlargest(5, 'score')[['gene', 'score', 'score_type', 'raw_count']].to_string(index=False))
    fig = plots.plot_sample(df, sid, pheno)
    fig.savefig(config.CV_FIG_DIR / f'score_{pheno.replace(" ", "_")}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# 전체 disease Z matrix → Z_disease/sample/gene .npy + |z|>=z_flag long parquet
X_dis = X_raw[dis_idx]
Y_dis = Y[dis_idx]
Z_full, flagged = scoring.score_full(engine, rare_scorer, gene_names,
                                     X_dis, Y_dis, dis_names, dis_pheno)
print(f'Z matrix: {Z_full.shape}   flagged rows: {len(flagged):,}')